# OMNI-904 / OMNI-X16 · The PFM on CCD: its proofs, then one model per clip encoder (Colab GPU)

Runs the repo's own code from a clone of the branch: `scripts/pfm_model.py` (Algorithm 4), `scripts/train_pfm.py`
(training, resume, the X16 measures) and `scripts/pfm_samples.py` (`datasets/samples/ccd.csv`, D-55).

1. **Setup:** clone, Drive
2. **The vectors:** both encoders' CCD folders copied from Drive to this machine (`embeddings/pfm/ccd/...`)
3. **Overfit proof** (OMNI-904): 32 training samples, the loss goes near zero
4. **Resume proof** (OMNI-904): 4 epochs straight, against 2 epochs, a restart from the checkpoint and 2 more
5. **X16:** the same config, 3 seeds per encoder, trained on CCD's train split, measured on **validation**;
   the ticket's rule → `MyDrive/omnisight/results/x16_pfm_ccd.json`

Checkpoints on Drive (`models/pfm/x16/...`): if the session ends, Run all again and every run resumes.
Runtime → Change runtime type → **GPU**. Held-out data is not touched.

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, "/content/omni-sight/scripts")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
DEVICE, GPU = "cuda", torch.cuda.get_device_name(0)
print(torch.__version__, GPU)

In [ ]:
# 2 · Both encoders' vectors, copied from Drive once, and the samples
import json, shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pfm_samples, train_pfm
from pfm_model import Config

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
ENCODERS = ("internvideo2", "cosmos")
SAMPLES = pfm_samples.read_samples(REPO_DIR / "datasets" / "samples" / "ccd.csv")
KEYS = sorted({s["clip_key"] for s in SAMPLES})

def copy_folder(name):
    source, target = DRIVE / "embeddings" / "pfm" / "ccd" / name, Path("/content/pfm") / name
    def one(key):
        dst = target / f"{key}.npz"
        if not dst.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / f"{key}.npz", dst)
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(one, KEYS))
    return target

STORES = {}
for name in ENCODERS:
    start = time.time()
    STORES[name] = pfm_samples.Store(SAMPLES, copy_folder(name))
    print(f"{name}: {len(STORES[name].vectors):,} vectors of {STORES[name].dim}-D, {time.time() - start:.0f} s")
print(f"{len(SAMPLES):,} samples")

In [ ]:
# 3 · Overfit proof: 32 training samples (16 Crash, 16 Normal, InternVideo2), the loss goes near zero
train_iv2 = train_pfm.Data.split(STORES["internvideo2"], SAMPLES, "train", "ccd")
rng = np.random.default_rng(904)
tiny = np.concatenate([rng.choice(np.flatnonzero(train_iv2.y == c), 16, replace=False) for c in (0, 1)])
tiny_data = train_iv2.subset(tiny)
overfit_config = Config(input_dim=512, domains=["ccd"], max_length=10, dropout=0.0, lr=3e-3, batch=32, epochs=300)
proof_dir = DRIVE / "models" / "pfm" / "x16" / "proofs"
shutil.rmtree(proof_dir / "overfit", ignore_errors=True)
model, history = train_pfm.train(overfit_config, {"ccd": tiny_data}, proof_dir / "overfit", seed=0, device=DEVICE, log=lambda m: None)
p, _ = train_pfm.predict(model, tiny_data, DEVICE)
OVERFIT = {"samples": 32, "epochs": 300, "first_loss": history[0]["loss"], "last_loss": history[-1]["loss"],
           "last_ce": history[-1]["ccd_ce"], "last_mse": history[-1]["ccd_mse"], "train_top1": float((p.argmax(1) == tiny_data.y).mean())}
print(OVERFIT)
assert OVERFIT["last_loss"] < 0.05 and OVERFIT["train_top1"] == 1.0

In [ ]:
# 4 · Resume proof: 4 epochs straight, against 2 epochs + a restart from the checkpoint + 2 more
resume_config = Config(input_dim=512, domains=["ccd"], max_length=10, epochs=4)
for name in ("straight", "resumed"):
    shutil.rmtree(proof_dir / name, ignore_errors=True)
_, straight = train_pfm.train(resume_config, {"ccd": train_iv2}, proof_dir / "straight", seed=0, device=DEVICE)
_, first = train_pfm.train(resume_config, {"ccd": train_iv2}, proof_dir / "resumed", seed=0, device=DEVICE, stop_after=2)
_, resumed = train_pfm.train(resume_config, {"ccd": train_iv2}, proof_dir / "resumed", seed=0, device=DEVICE)  # a new model, from last.pt
a, b = np.array([h["loss"] for h in straight]), np.array([h["loss"] for h in resumed])
RESUME = {"straight": a.tolist(), "resumed": b.tolist(), "stopped_after": len(first), "max_relative_difference": float(np.max(np.abs(a - b) / a))}
print(RESUME)
assert len(first) == 2 and len(resumed) == 4 and RESUME["max_relative_difference"] < 1e-3

In [ ]:
# 5 · X16: the same config for both encoders (only the input size differs), 3 seeds, measured on validation
import hashlib
SEEDS = (0, 1, 2)
RESULTS, CONFIGS = {}, {}
for name in ENCODERS:
    store = STORES[name]
    train_data = train_pfm.Data.split(store, SAMPLES, "train", "ccd")
    val_data = train_pfm.Data.split(store, SAMPLES, "validation", "ccd")
    config = Config(input_dim=store.dim, domains=["ccd"], max_length=10)
    CONFIGS[name], RESULTS[name] = config.to_dict(), []
    for seed in SEEDS:
        out = DRIVE / "models" / "pfm" / "x16" / name / f"seed{seed}"
        model, history = train_pfm.train(config, {"ccd": train_data}, out, seed=seed, device=DEVICE, log=lambda m: None)
        p, dt = train_pfm.predict(model, val_data, DEVICE)
        m = train_pfm.evaluate(val_data, p, dt)
        m.update(seed=seed, final_train_loss=history[-1]["loss"], train_seconds=sum(h["seconds"] for h in history))
        RESULTS[name].append(m)
        print(f"{name:13s} seed {seed}: AUC {m['roc_auc']:.4f}  top-1 {m['top1']:.4f}  Δt MAE {m['dt_mae_s']:.3f} s  "
              f"lead {m['lead_time_s'] or 0:.2f} s ({m['warned_windows']}/{m['positive_windows']} warned, θ {m['theta']:.3f}, "
              f"false alarms {m['false_alarm_windows']:.3f})  {m['train_seconds']:.0f} s")

RULE = train_pfm.x16_rule(RESULTS)
for name, s in RULE["summary"].items():
    print(f"{name:13s} AUC {s['auc_mean']:.4f} (spread {s['auc_spread']:.4f})  lead {s['lead_mean_s']:.2f} s  "
          f"top-1 {s['top1_mean']:.4f}  Δt MAE {s['dt_mae_s_mean']:.3f} s")
print(f"margin {RULE['margin']:.4f} -> the PFM reads: {RULE['winner']}")

out = DRIVE / "results" / "x16_pfm_ccd.json"
out.write_text(json.dumps({"commit": COMMIT, "gpu": GPU, "torch": torch.__version__, "samples_sha": hashlib.sha256(
    (REPO_DIR / "datasets" / "samples" / "ccd.csv").read_bytes()).hexdigest(), "overfit": OVERFIT, "resume": RESUME,
    "configs": CONFIGS, "seeds": SEEDS, "false_alarm_rule": train_pfm.FALSE_ALARM, "results": RESULTS, "rule": RULE}, indent=1))
print(out)